# Assistiv Systems — Rural Access Vulnerability Index (RAVI) v1.4
## Master Pipeline

Scores all 1,065 Kent & Medway LSOAs for rural access vulnerability and commits `kent-ravi-data.json` to GitHub with precise ONS population-weighted centroids.

**Purpose:** Reveal hidden vulnerability in apparently low-risk districts — isolated older adults in rural Sevenoaks, the Weald, Romney Marsh, and Ashford edges — invisible to district-level FEP scores.

**Five signals — all open data, no API key:**
| Signal | Source | Weight |
|---|---|---|
| Geographic Barriers to Services | IMD 2019 File 7 (MHCLG) | 30% |
| Rural Urban Classification | mySociety/ONS 2021 | 25% |
| Population aged 65+ | Census 2021 via Nomis | 20% |
| No car or van in household | Census 2021 via Nomis | 15% |
| Day-to-day limited a lot | Census 2021 via Nomis | 10% |

**Run order:**
1. **Cell 1** — Install dependencies
2. **Cell 2** — Fetch data, calculate RAVI, commit JSON
3. **Cell 3** — Fetch precise ONS centroids and update JSON

**Before running:** Add `GITHUB_TOKEN` to Colab Secrets (🔑 left sidebar)

---
*Sources: MHCLG IMD 2019 · mySociety/ONS RUC 2021 · Census 2021 Nomis · ONS LSOA Centroids 2021*

## Cell 1 — Install Dependencies

In [ ]:
!pip install requests pandas openpyxl beautifulsoup4 --quiet
print("Dependencies installed ✓")

## Cell 2 — Data Pipeline
Fetches IMD 2019 geographic barriers, ONS Rural Urban Classification, and Census 2021 signals.
Calculates RAVI composite scores for all 1,065 Kent LSOAs and commits `kent-ravi-data.json`.

In [ ]:
"""
RAVI Pipeline v1.4 — Data fetch, score calculation, GitHub commit
"""

import requests, pandas as pd, json, base64, io
from datetime import datetime, timezone
from google.colab import userdata

GITHUB_REPO  = "silegrand/assistivagents"
GITHUB_FILE  = "kent-ravi-data.json"
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN').split('\n')[0].strip()

KENT_LAD_CODES = [
    'E07000105','E07000106','E07000107','E07000108','E07000109',
    'E07000110','E06000035','E07000111','E07000112','E07000113',
    'E07000114','E07000115','E07000116',
]

WEIGHTS = {
    'geo_barriers': 0.30,
    'ruc_score':    0.25,
    'pct_65plus':   0.20,
    'pct_no_car':   0.15,
    'pct_llti':     0.10,
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 0.001
print(f"Weight check: {sum(WEIGHTS.values()):.2f} ✓")

LAD_NAMES = {
    'E07000105':'Ashford',            'E07000106':'Canterbury',
    'E07000107':'Dartford',           'E07000108':'Dover',
    'E07000109':'Gravesham',          'E07000110':'Maidstone',
    'E06000035':'Medway',             'E07000111':'Sevenoaks',
    'E07000112':'Folkestone & Hythe', 'E07000113':'Swale',
    'E07000114':'Thanet',             'E07000115':'Tonbridge & Malling',
    'E07000116':'Tunbridge Wells',
}

# England Census 2021 national averages — fallback if Nomis unavailable
ENG_FALLBACKS = {'pct_65plus': 18.4, 'pct_no_car': 25.6, 'pct_llti': 17.8}

HEADERS = {"User-Agent": "Mozilla/5.0 AssistivSystems/1.0"}


# ── STEP 1: IMD 2019 GEOGRAPHIC BARRIERS ─────────────────────────────
print("\n── Step 1: IMD 2019 Geographic Barriers ──")
IMD_URL = (
    "https://assets.publishing.service.gov.uk/media/5dc407b440f0b6379a7acc8d/"
    "File_7_-_All_IoD2019_Scores__Ranks__Deciles_and_Population_Denominators_3.csv"
)
r = requests.get(IMD_URL, timeout=120, headers=HEADERS)
print(f"  HTTP {r.status_code} | {len(r.content):,} bytes")
if r.status_code != 200:
    raise RuntimeError(
        "IMD download failed. Download File 7 CSV manually from:\n"
        "https://www.gov.uk/government/statistics/english-indices-of-deprivation-2019"
    )

imd_raw = pd.read_csv(io.StringIO(r.text))
print(f"  {len(imd_raw):,} rows | {len(imd_raw.columns)} columns")

lsoa_col  = 'LSOA code (2011)'
lsoa_name = 'LSOA name (2011)'
lad_col   = 'Local Authority District code (2019)'
geo_col   = next((c for c in imd_raw.columns if 'Geographical Barriers' in c and 'Score' in c), None)
if not geo_col:
    geo_col = next((c for c in imd_raw.columns if 'Geograph' in c and 'Score' in c), None)
if not geo_col:
    print("Columns:", list(imd_raw.columns))
    raise ValueError("Cannot find Geographic Barriers Score column")
print(f"  Geo barriers column: {geo_col!r}")

imd_kent = imd_raw[imd_raw[lad_col].isin(KENT_LAD_CODES)][
    [lsoa_col, lsoa_name, lad_col, geo_col]
].copy()
imd_kent.columns = ['lsoa_code','lsoa_name','lad_code','geo_barriers_score']
imd_kent = imd_kent.dropna(subset=['geo_barriers_score']).reset_index(drop=True)
print(f"  Kent LSOAs: {len(imd_kent)}")
print(f"  Score range: {imd_kent.geo_barriers_score.min():.3f} – {imd_kent.geo_barriers_score.max():.3f}")


# ── STEP 2: ONS 2021 RURAL URBAN CLASSIFICATION ───────────────────────
print("\n── Step 2: Rural Urban Classification ──")
RUC_URL = "https://raw.githubusercontent.com/mysociety/uk_ruc/main/output/composite_ruc.csv"
r = requests.get(RUC_URL, timeout=30, headers=HEADERS)
print(f"  HTTP {r.status_code} | {len(r.content):,} bytes")

if r.status_code == 200:
    ruc_raw  = pd.read_csv(io.StringIO(r.text))
    ruc_col  = 'ukruc-3' if 'ukruc-3' in ruc_raw.columns else 'ukruc-2'
    print(f"  Using column: {ruc_col!r} | Values: {sorted(ruc_raw[ruc_col].unique())}")

    if ruc_col == 'ukruc-3':
        code_map  = {0:'Urban', 1:'Rural town/village', 2:'Hamlet/Isolated'}
        score_map = {0:0, 1:55, 2:90}
    else:
        code_map  = {0:'Urban', 1:'Rural'}
        score_map = {0:10, 1:70}

    ruc_kent = ruc_raw[ruc_raw['lsoa'].isin(imd_kent.lsoa_code)][['lsoa', ruc_col]].copy()
    ruc_kent.columns = ['lsoa_code', 'ruc_raw']
    ruc_kent['ruc_label'] = ruc_kent.ruc_raw.map(code_map).fillna('Unknown')
    ruc_kent['ruc_score'] = ruc_kent.ruc_raw.map(score_map).fillna(30.0)
    imd_kent = imd_kent.merge(ruc_kent[['lsoa_code','ruc_label','ruc_score']], on='lsoa_code', how='left')
    imd_kent['ruc_label'] = imd_kent['ruc_label'].fillna('Unknown')
    imd_kent['ruc_score'] = imd_kent['ruc_score'].fillna(30.0)
    matched = imd_kent['ruc_label'].ne('Unknown').sum()
    print(f"  Matched: {matched}/{len(imd_kent)} Kent LSOAs")
    print(f"  Distribution:\n{imd_kent.ruc_label.value_counts().to_string()}")
else:
    print("  WARNING: RUC unavailable — assigning 30 to all LSOAs")
    imd_kent['ruc_label'] = 'Unknown'
    imd_kent['ruc_score'] = 30.0


# ── STEP 3: CENSUS 2021 VIA NOMIS ────────────────────────────────────
print("\n── Step 3: Census 2021 via Nomis ──")

def fetch_nomis(dataset_id, extra_params, description, col_name, fallback):
    url = (
        f"https://www.nomisweb.co.uk/api/v01/dataset/{dataset_id}.data.csv"
        f"?date=latest&geography=TYPE297&measures=20301"
        f"&{extra_params}&select=GEOGRAPHY_CODE,OBS_VALUE"
    )
    print(f"  {description}...")
    try:
        r = requests.get(url, timeout=60, headers=HEADERS)
        print(f"    HTTP {r.status_code} | {len(r.content):,} bytes")
        if r.status_code == 200 and len(r.content) > 500:
            df = pd.read_csv(io.StringIO(r.text))
            df.columns = ['lsoa_code', col_name]
            df = df[df.lsoa_code.isin(imd_kent.lsoa_code)]
            if len(df) > 100:
                df = df.groupby('lsoa_code')[col_name].sum().reset_index()
                print(f"    ✓ {len(df)} LSOAs | Mean: {df[col_name].mean():.1f}%")
                return df
        print(f"    Using England average {fallback}%")
    except Exception as e:
        print(f"    Error: {e} — using England average {fallback}%")
    return None

age_df  = fetch_nomis('NM_2010_1', 'c2021_age_h=7,8,9,10,11,12,13,14,15,16,17,18',
                       'Population aged 65+ %', 'pct_65plus', ENG_FALLBACKS['pct_65plus'])
car_df  = fetch_nomis('NM_2072_1', 'c2021_carsvan_4=1',
                       'No car access %', 'pct_no_car', ENG_FALLBACKS['pct_no_car'])
llti_df = fetch_nomis('NM_2064_1', 'c2021_disability_3=1',
                       'Day-to-day limited a lot %', 'pct_llti', ENG_FALLBACKS['pct_llti'])

for df, col in [(age_df,'pct_65plus'), (car_df,'pct_no_car'), (llti_df,'pct_llti')]:
    fallback = ENG_FALLBACKS[col]
    if df is not None:
        imd_kent = imd_kent.merge(df, on='lsoa_code', how='left')
        imd_kent[col] = imd_kent[col].fillna(fallback)
    else:
        imd_kent[col] = fallback

print(f"\nDataset ready: {imd_kent.shape[0]} LSOAs × {imd_kent.shape[1]} columns")


# ── STEP 4: NORMALISE AND SCORE ────────────────────────────────────────
print("\n── Step 4: Normalise and calculate RAVI ──")

def norm_to_100(series):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series([50.0] * len(series), index=series.index)
    return ((series - mn) / (mx - mn) * 100).round(1)

imd_kent['geo_barriers_norm'] = norm_to_100(imd_kent.geo_barriers_score)
imd_kent['ruc_norm']          = norm_to_100(imd_kent.ruc_score)
imd_kent['age_norm']          = norm_to_100(imd_kent.pct_65plus)
imd_kent['car_norm']          = norm_to_100(imd_kent.pct_no_car)
imd_kent['llti_norm']         = norm_to_100(imd_kent.pct_llti)

imd_kent['ravi'] = (
    imd_kent.geo_barriers_norm * WEIGHTS['geo_barriers'] +
    imd_kent.ruc_norm          * WEIGHTS['ruc_score']    +
    imd_kent.age_norm          * WEIGHTS['pct_65plus']   +
    imd_kent.car_norm          * WEIGHTS['pct_no_car']   +
    imd_kent.llti_norm         * WEIGHTS['pct_llti']
).round(1)

def ravi_band(s):
    if s >= 70: return 'critical'
    if s >= 55: return 'high'
    if s >= 40: return 'moderate'
    return 'low'

imd_kent['ravi_band'] = imd_kent.ravi.apply(ravi_band)
imd_kent['district']  = imd_kent.lad_code.map(LAD_NAMES)

print(f"RAVI: {imd_kent.ravi.min():.1f} – {imd_kent.ravi.max():.1f} | Mean: {imd_kent.ravi.mean():.1f}")
print(f"\nRisk bands:\n{imd_kent.ravi_band.value_counts().to_string()}")
print(f"\nTop 20 most vulnerable LSOAs:")
top = imd_kent.nlargest(20,'ravi')[['lsoa_name','district','ravi','ravi_band','ruc_label','geo_barriers_score']]
print(top.to_string(index=False))


# ── STEP 5: DISTRICT SUMMARY ──────────────────────────────────────────
print("\n── Step 5: District summary ──")
print(f"{'District':<25} {'Mean':>5}  {'Max':>5}  {'High/Crit':>9}  Top LSOA")
print("-" * 80)
district_summary = {}
for dist, grp in imd_kent.groupby('district'):
    high = grp[grp.ravi >= 55]
    district_summary[dist] = {
        'lsoa_count':      int(len(grp)),
        'mean_ravi':       round(float(grp.ravi.mean()), 1),
        'max_ravi':        round(float(grp.ravi.max()), 1),
        'high_ravi_count': int(len(high)),
        'pct_high':        round(len(high)/len(grp)*100, 1),
        'top_lsoa':        str(grp.nlargest(1,'ravi').iloc[0]['lsoa_name']),
    }
    s = district_summary[dist]
    print(f"  {dist:<23} {s['mean_ravi']:>5.1f}  {s['max_ravi']:>5.1f}  {s['high_ravi_count']:>9}  {s['top_lsoa'][:35]}")


# ── STEP 6: BUILD AND COMMIT JSON ─────────────────────────────────────
print("\n── Step 6: Commit JSON (coordinates added by Cell 3) ──")

# Placeholder coordinates — Cell 3 replaces with precise ONS centroids
DISTRICT_CENTRES = {
    'Ashford':(51.148,0.874),'Canterbury':(51.279,1.076),'Dartford':(51.446,0.221),
    'Dover':(51.126,1.313),'Folkestone & Hythe':(51.083,1.167),'Gravesham':(51.442,0.368),
    'Maidstone':(51.272,0.523),'Medway':(51.385,0.523),'Sevenoaks':(51.272,0.187),
    'Swale':(51.333,0.753),'Thanet':(51.358,1.383),'Tonbridge & Malling':(51.197,0.273),
    'Tunbridge Wells':(51.132,0.263),
}
dist_count = {}
for _, row in imd_kent.iterrows():
    d   = row.district
    idx = dist_count.get(d, 0)
    dist_count[d] = idx + 1
    dc  = DISTRICT_CENTRES.get(d, (51.27, 0.52))
    imd_kent.at[_, 'lat'] = round(dc[0] - 0.07 + (idx // 10) * 0.016, 5)
    imd_kent.at[_, 'lon'] = round(dc[1] - 0.07 + (idx  % 10) * 0.016, 5)

lsoa_list = [{
    'lsoa_code': row.lsoa_code,
    'lsoa_name': str(row.lsoa_name),
    'district':  str(row.get('district','')),
    'lad_code':  row.lad_code,
    'lat':       round(float(row.get('lat', 51.27)), 5),
    'lon':       round(float(row.get('lon', 0.52)), 5),
    'ravi':      round(float(row.ravi), 1),
    'ravi_band': row.ravi_band,
    'signals': {
        'geo_barriers_score': round(float(row.geo_barriers_score), 3),
        'geo_barriers_norm':  round(float(row.geo_barriers_norm), 1),
        'ruc_label':          str(row.get('ruc_label','Unknown')),
        'ruc_score':          round(float(row.get('ruc_score',30)), 1),
        'ruc_norm':           round(float(row.ruc_norm), 1),
        'pct_65plus':         round(float(row.pct_65plus), 1),
        'age_norm':           round(float(row.age_norm), 1),
        'pct_no_car':         round(float(row.pct_no_car), 1),
        'car_norm':           round(float(row.car_norm), 1),
        'pct_llti':           round(float(row.pct_llti), 1),
        'llti_norm':          round(float(row.llti_norm), 1),
    }
} for _, row in imd_kent.iterrows()]

output = {
    'meta': {
        'generated':   datetime.now(timezone.utc).isoformat(),
        'description': 'Rural Access Vulnerability Index — Kent & Medway LSOAs',
        'version':     '1.4',
        'lsoa_count':  len(lsoa_list),
        'sources': {
            'geo_barriers': 'IMD 2019 File 7 — Geographic Barriers sub-domain (MHCLG)',
            'ruc':          'mySociety composite RUC 2021',
            'census':       'Census 2021 via Nomis API or England average fallback',
        },
        'weights':           WEIGHTS,
        'census_source':     'real' if age_df is not None else 'england_average_fallback',
        'coord_method':      'district grid placeholder — run Cell 3 for precise centroids',
        'has_coordinates':   True,
    },
    'district_summary': district_summary,
    'lsoas':            lsoa_list,
}

def commit_file(content, filepath, token, message):
    api  = f"https://api.github.com/repos/{GITHUB_REPO}/contents/{filepath}"
    hdrs = {"Authorization":f"token {token}","Accept":"application/vnd.github.v3+json"}
    b64  = base64.b64encode(json.dumps(content,indent=2).encode()).decode()
    r    = requests.get(api, headers=hdrs)
    sha  = r.json().get("sha") if r.status_code == 200 else None
    pay  = {"message":message,"content":b64,"branch":"main"}
    if sha: pay["sha"] = sha
    r = requests.put(api, headers=hdrs, json=pay)
    if r.status_code in (200,201):
        print(f"  ✓ {filepath}")
        return True
    print(f"  ✗ {r.status_code} — {r.json().get('message','')}")
    return False

msg = f"RAVI v1.4 — {datetime.now(timezone.utc).strftime('%Y-%m-%d')} — {len(lsoa_list)} Kent LSOAs"
commit_file(output, GITHUB_FILE, GITHUB_TOKEN, msg)
print(f"\nCell 2 complete — run Cell 3 to add precise ONS centroids")
print(f"High/critical LSOAs: {sum(1 for l in lsoa_list if l['ravi_band'] in ['high','critical'])}")


## Cell 3 — Add Precise ONS Centroids
Fetches ONS 2021 population-weighted centroids via ArcGIS geometry extraction.
Updates `kent-ravi-data.json` with precise lat/lon for each LSOA.
Run after Cell 2. Safe to re-run — overwrites coordinates only.

In [ ]:
"""
Fetch ONS LSOA population-weighted centroids — geometry extraction from ArcGIS.
Replaces grid approximation coordinates with precise positions.
"""

import requests, json, base64
from datetime import datetime, timezone
from google.colab import userdata

GITHUB_REPO  = "silegrand/assistivagents"
GITHUB_FILE  = "kent-ravi-data.json"
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN').split('\n')[0].strip()
HEADERS      = {"User-Agent": "Mozilla/5.0 AssistivSystems/1.0"}

# Load current RAVI JSON
print("Loading current kent-ravi-data.json...")
r = requests.get(
    f"https://raw.githubusercontent.com/{GITHUB_REPO}/main/{GITHUB_FILE}",
    timeout=15
)
ravi       = r.json()
kent_codes = set(l['lsoa_code'] for l in ravi['lsoas'])
print(f"  {len(kent_codes)} Kent LSOA codes | Current version: {ravi['meta'].get('version')}")

# ── FETCH CENTROIDS FROM ONS ARCGIS ──────────────────────────────────
# Coordinates are in geometry object (not attribute fields) — use returnGeometry=true
print("\n── Fetching ONS LSOA centroids via geometry extraction ──")

BASE = ("https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/"
        "LSOA_PopCentroids_EW_2021_V4/FeatureServer/0/query")

all_centroids = {}
offset = 0
batch  = 1000
errors = 0

print("  Fetching all England/Wales LSOAs in batches of 1,000...")
while True:
    r = requests.get(BASE, timeout=30, headers=HEADERS, params={
        'where':             '1=1',
        'outFields':         'LSOA21CD',
        'returnGeometry':    'true',
        'outSR':             '4326',
        'f':                 'json',
        'resultOffset':      offset,
        'resultRecordCount': batch,
    })
    if r.status_code != 200:
        errors += 1
        if errors > 3: break
        continue

    data = r.json()
    if 'error' in data:
        print(f"  API error: {data['error']}")
        break

    features = data.get('features', [])
    if not features:
        break

    for f in features:
        code = f.get('attributes', {}).get('LSOA21CD')
        geom = f.get('geometry', {})
        if code and geom and geom.get('x') and geom.get('y'):
            all_centroids[code] = {
                'lat': round(float(geom['y']), 5),
                'lon': round(float(geom['x']), 5),
            }

    offset += len(features)
    if offset % 5000 == 0:
        print(f"  {offset:,} fetched...")
    if len(features) < batch:
        break

print(f"  Total fetched: {offset:,} | With coordinates: {len(all_centroids):,}")

# ── MATCH TO KENT ─────────────────────────────────────────────────────
kent_matched = {code: all_centroids[code]
                for code in kent_codes
                if code in all_centroids}
match_pct = len(kent_matched) / len(kent_codes) * 100
print(f"  Kent matched: {len(kent_matched)}/{len(kent_codes)} ({match_pct:.0f}%)")

if match_pct < 50:
    print("\nWARNING: Less than 50% matched — 2011/2021 boundary differences")
    print("The existing coordinates will be kept.")
    print("Download LSOA 2011 centroids from https://geoportal.statistics.gov.uk/")
    raise SystemExit("Insufficient matches")

# ── APPLY AND VERIFY ──────────────────────────────────────────────────
updated = 0
for lsoa in ravi['lsoas']:
    if lsoa['lsoa_code'] in kent_matched:
        lsoa['lat'] = kent_matched[lsoa['lsoa_code']]['lat']
        lsoa['lon'] = kent_matched[lsoa['lsoa_code']]['lon']
        updated += 1

ravi['meta']['version']      = '1.4'
ravi['meta']['coord_method'] = 'ONS population-weighted centroids — geometry extraction'
ravi['meta']['generated']    = datetime.now(timezone.utc).isoformat()

print(f"  Updated: {updated}/{len(ravi['lsoas'])} LSOAs with precise coordinates")

# Sanity check
sevenoaks = [l for l in ravi['lsoas'] if l['district'] == 'Sevenoaks'][:2]
thanet    = [l for l in ravi['lsoas'] if l['district'] == 'Thanet'][:2]
print(f"\nSevenoaks sample (expect ~51.2-51.4, 0.1-0.3):")
for l in sevenoaks: print(f"  {l['lsoa_name']}: {l['lat']}, {l['lon']}")
print(f"Thanet sample (expect ~51.3-51.4, 1.25-1.42):")
for l in thanet: print(f"  {l['lsoa_name']}: {l['lat']}, {l['lon']}")

# ── COMMIT ────────────────────────────────────────────────────────────
def commit_file(content, filepath, token, message):
    api  = f"https://api.github.com/repos/{GITHUB_REPO}/contents/{filepath}"
    hdrs = {"Authorization":f"token {token}","Accept":"application/vnd.github.v3+json"}
    b64  = base64.b64encode(json.dumps(content,indent=2).encode()).decode()
    r    = requests.get(api, headers=hdrs)
    sha  = r.json().get("sha") if r.status_code == 200 else None
    pay  = {"message":message,"content":b64,"branch":"main"}
    if sha: pay["sha"] = sha
    r = requests.put(api, headers=hdrs, json=pay)
    if r.status_code in (200,201):
        print(f"  ✓ {filepath}")
        return True
    print(f"  ✗ {r.status_code} — {r.json().get('message','')}")
    return False

today = datetime.now(timezone.utc).strftime('%Y-%m-%d')
print("\nCommitting...")
commit_file(ravi, GITHUB_FILE, GITHUB_TOKEN,
            f"RAVI v1.4 — {today} — precise ONS centroids ({updated} LSOAs)")
print(f"\nDone — {updated} LSOAs with precise coordinates")
print(f"View: https://silegrand.github.io/assistivagents/rural-access-vulnerability.html")


## Cell 4 — Add Place Names
Maps each LSOA to its settlement name using the ONS LSOA to Built-Up Area (2011) lookup.
Rural LSOAs with no built-up area get 'Rural (no settlement)' — confirming genuine isolation.
Run after Cell 3. Adds `place_name` field to every LSOA in the JSON.

In [ ]:
"""
Add ONS Built-Up Area place names to RAVI JSON.
Maps LSOA codes to settlement names (village, town, city).
Rural/isolated LSOAs with no BUA get 'Rural (no settlement)'.
"""

import requests, json, base64, pandas as pd, io
from datetime import datetime, timezone
from google.colab import userdata

GITHUB_REPO  = "silegrand/assistivagents"
GITHUB_FILE  = "kent-ravi-data.json"
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN').split('\n')[0].strip()
HEADERS      = {"User-Agent": "Mozilla/5.0 AssistivSystems/1.0"}

# Load current RAVI JSON
print("Loading kent-ravi-data.json...")
r    = requests.get(f"https://raw.githubusercontent.com/{GITHUB_REPO}/main/{GITHUB_FILE}", timeout=15)
ravi = r.json()
kent_codes = set(l['lsoa_code'] for l in ravi['lsoas'])
print(f"  {len(kent_codes)} Kent LSOAs | Version: {ravi['meta'].get('version')}")

# ── FETCH ONS LSOA TO BUILT-UP AREA LOOKUP ───────────────────────────
# LSOA (2011) to BUASD to BUA to LAD to RGN (December 2011) Best Fit Lookup EW
# Fields: LSOA11CD, BUASD11NM (sub-division), BUA11NM (built-up area)
# Rural LSOAs have blank BUA fields — these are the genuinely isolated ones
print("\n── Fetching ONS LSOA to Built-Up Area lookup ──")

BUA_URLS = [
    # ONS Open Geography Portal CSV download
    "https://opendata.arcgis.com/api/v3/datasets/a9ef1b7b695843e8923d1b1f7ea06591_0/downloads/data?format=csv&spatialRefId=4326",
    # ArcGIS FeatureServer JSON — geometry not needed, attributes only
    "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/LSOA11_BUASD11_BUA11_LAD11_RGN11_EW_LU/FeatureServer/0/query?where=1%3D1&outFields=LSOA11CD,BUASD11NM,BUA11NM&f=json&resultRecordCount=40000",
    # Alternative open data URL
    "https://opendata.arcgis.com/datasets/a9ef1b7b695843e8923d1b1f7ea06591_0.csv",
]

bua_df = None
for url in BUA_URLS:
    print(f"  Trying {url[-60:]}...")
    try:
        r = requests.get(url, timeout=30, headers=HEADERS)
        print(f"  HTTP {r.status_code} | {len(r.content):,} bytes")
        if r.status_code != 200 or len(r.content) < 10000:
            continue
        if 'f=json' in url:
            data = r.json()
            if 'error' in data:
                print(f"  Error: {data['error']}")
                continue
            features = data.get('features', [])
            if features:
                bua_df = pd.DataFrame([f['attributes'] for f in features])
                print(f"  ✓ Loaded {len(bua_df):,} records (JSON)")
                break
        else:
            bua_df = pd.read_csv(io.StringIO(r.text))
            print(f"  ✓ Loaded {len(bua_df):,} records (CSV)")
            print(f"  Columns: {list(bua_df.columns[:8])}")
            break
    except Exception as e:
        print(f"  Error: {e}")

if bua_df is None:
    print("\nPlace name lookup unavailable from all sources.")
    print("Manual option:")
    print("  1. Go to https://geoportal.statistics.gov.uk/")
    print("  2. Search: 'LSOA 2011 to Built-up Area 2011 Best Fit Lookup'")
    print("  3. Download CSV, upload to Colab")
    print("  4. Run: bua_df = pd.read_csv('your_file.csv')")
    print("     Then continue from the matching section below")
    raise SystemExit("BUA lookup unavailable")

# ── IDENTIFY COLUMNS ──────────────────────────────────────────────────
lsoa_col  = next((c for c in bua_df.columns if 'LSOA11CD' in str(c).upper()), None)
bua_col   = next((c for c in bua_df.columns if 'BUA11NM' in str(c).upper()
                  and 'BUASD' not in str(c).upper()), None)
buasd_col = next((c for c in bua_df.columns if 'BUASD11NM' in str(c).upper()), None)

if not lsoa_col:
    print(f"All columns: {list(bua_df.columns)}")
    raise ValueError("Cannot find LSOA11CD column")

print(f"\n  LSOA col:  {lsoa_col!r}")
print(f"  BUA col:   {bua_col!r}")
print(f"  BUASD col: {buasd_col!r}")

# ── FILTER TO KENT AND BUILD LOOKUP ──────────────────────────────────
kent_bua = bua_df[bua_df[lsoa_col].isin(kent_codes)].copy()

if buasd_col and bua_col:
    kent_bua = kent_bua[[lsoa_col, buasd_col, bua_col]].copy()
    kent_bua.columns = ['lsoa_code', 'buasd_name', 'bua_name']
elif bua_col:
    kent_bua = kent_bua[[lsoa_col, bua_col]].copy()
    kent_bua.columns = ['lsoa_code', 'bua_name']
    kent_bua['buasd_name'] = None
else:
    print(f"Columns: {list(bua_df.columns)}")
    raise ValueError("Cannot find BUA name columns")

def place_name(row):
    """Priority: BUASD (most specific) > BUA > Rural label"""
    buasd = str(row.get('buasd_name', '') or '').strip()
    bua   = str(row.get('bua_name',   '') or '').strip()
    if buasd and buasd not in ('nan', 'None', ''):
        return buasd
    if bua and bua not in ('nan', 'None', ''):
        return bua
    return 'Rural (no settlement)'

kent_bua['place_name'] = kent_bua.apply(place_name, axis=1)
bua_lookup = dict(zip(kent_bua.lsoa_code, kent_bua.place_name))

n_named   = sum(1 for v in bua_lookup.values() if v != 'Rural (no settlement)')
n_rural   = sum(1 for v in bua_lookup.values() if v == 'Rural (no settlement)')
n_matched = len(bua_lookup)
print(f"\n  Matched: {n_matched}/{len(kent_codes)} Kent LSOAs")
print(f"  Named settlements: {n_named}")
print(f"  Rural (no settlement): {n_rural}")

# ── UPDATE RAVI JSON ──────────────────────────────────────────────────
added = 0
for lsoa in ravi['lsoas']:
    lsoa['place_name'] = bua_lookup.get(lsoa['lsoa_code'], 'Unknown')
    # IMPORTANT: preserve existing coordinates — do not overwrite lat/lon
    added += 1

ravi['meta']['has_place_names'] = True
ravi['meta']['generated']       = datetime.now(timezone.utc).isoformat()

# Show top critical LSOAs with place names
print(f"\n── Top 15 critical RAVI LSOAs with place names ──")
print(f"{'LSOA':<25} {'Place':<30} {'District':<22} {'RAVI':>5}")
print("-" * 85)
critical = sorted(
    [l for l in ravi['lsoas'] if l['ravi_band'] in ('critical', 'high')],
    key=lambda x: -x['ravi']
)[:15]
for l in critical:
    print(f"  {l['lsoa_name']:<23} {l['place_name']:<30} {l['district']:<22} {l['ravi']:>5.1f}")

# Check rural breakdown by district
print(f"\n── Rural (no settlement) LSOAs by district ──")
rural_by_dist = {}
for l in ravi['lsoas']:
    if l['place_name'] == 'Rural (no settlement)':
        d = l['district']
        rural_by_dist[d] = rural_by_dist.get(d, 0) + 1
for d, n in sorted(rural_by_dist.items(), key=lambda x: -x[1]):
    print(f"  {d:<25} {n}")

# ── COMMIT ────────────────────────────────────────────────────────────
def commit_file(content, filepath, token, message):
    api  = f"https://api.github.com/repos/{GITHUB_REPO}/contents/{filepath}"
    hdrs = {"Authorization":f"token {token}","Accept":"application/vnd.github.v3+json"}
    b64  = base64.b64encode(json.dumps(content,indent=2).encode()).decode()
    r    = requests.get(api, headers=hdrs)
    sha  = r.json().get("sha") if r.status_code == 200 else None
    pay  = {"message":message,"content":b64,"branch":"main"}
    if sha: pay["sha"] = sha
    r = requests.put(api, headers=hdrs, json=pay)
    if r.status_code in (200,201):
        print(f"  ✓ {filepath}")
        return True
    print(f"  ✗ {r.status_code} — {r.json().get('message','')}")
    return False

today = datetime.now(timezone.utc).strftime('%Y-%m-%d')
print("\nCommitting...")
commit_file(ravi, GITHUB_FILE, GITHUB_TOKEN,
            f"RAVI — {today} — place names added ({n_named} settlements, {n_rural} rural)")
print(f"\nDone — place names added to {added} LSOAs")
